In [0]:
dbutils.widgets.text("p_data_source", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configs"

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
laptimes_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                     StructField("driverId", IntegerType(), True),
                                     StructField("lap", IntegerType(), True),
                                     StructField("position", IntegerType(), True),
                                     StructField("time", StringType(), True),
                                     StructField("milliseconds", IntegerType(), True)])

In [0]:
laptimes_df = spark.read.schema(laptimes_schema).csv(f"{raw_folder_path}/{v_file_date}/lap_times")

In [0]:
display(laptimes_df)

raceId,driverId,lap,position,time,milliseconds
1053,830,1,1,1:38.603,98603
1053,830,2,1,2:29.163,149163
1053,830,3,1,2:23.247,143247
1053,830,4,1,2:20.332,140332
1053,830,5,1,2:25.691,145691
1053,830,6,1,2:20.804,140804
1053,830,7,1,1:36.303,96303
1053,830,8,1,1:32.925,92925
1053,830,9,1,1:30.953,90953
1053,830,10,1,1:30.130,90130


In [0]:
laptimes_df.count()

1124

In [0]:
%run "../Includes/comm_func"  

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
laptimes_final_df = laptimes_df.withColumnRenamed("raceId", "race_id").withColumnRenamed("driverId", "driver_id").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source)).withColumn("file_date", lit(v_file_date))

In [0]:
laptimes_final_df = laptimes_final_df.dropDuplicates(
    ["race_id", "driver_id", "lap"]
)

In [0]:
laptimes_final_df.write \
    .mode("append") \
    .format("delta") \
    .partitionBy("race_id") \
    .save(f"{processed_folder_path}/laptimes")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS f1_processed.lap_times (
  race_id INT,
  driver_id INT,
  lap INT,
  position INT,
  time STRING,
  milliseconds INT,
  ingestion_date TIMESTAMP,
  file_date DATE,
  data_source STRING
)
USING DELTA
PARTITIONED BY (file_date);

In [0]:
#laptimes_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("f1_processed.lap_times")

In [0]:
%sql
--DROP TABLE IF EXISTS f1_processed.lap_times;

In [0]:
merge_into_table(
    laptimes_final_df,
    "f1_processed",
    "lap_times",
    """
    target.race_id = source.race_id
    AND target.driver_id = source.driver_id
    AND target.lap = source.lap
    """
)

In [0]:
%sql
SELECT * FROM f1_processed.lap_times
ORDER BY race_id DESC; 

race_id,driver_id,lap,position,time,milliseconds,ingestion_date,data_source,file_date
1053,830,1,1,1:38.603,98603,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,2,1,2:29.163,149163,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,3,1,2:23.247,143247,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,4,1,2:20.332,140332,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,5,1,2:25.691,145691,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,6,1,2:20.804,140804,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,7,1,1:36.303,96303,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,8,1,1:32.925,92925,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,9,1,1:30.953,90953,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18
1053,830,10,1,1:30.130,90130,2026-01-21T05:53:46.201499Z,Ergast API,2021-04-18


In [0]:
%sql
SELECT race_id, COUNT(race_id) FROM f1_processed.lap_times 
GROUP BY race_id
ORDER BY race_id DESC;

race_id,count(race_id)
1053,1124
1052,1026
1047,1043
1046,1531
1045,1016
1044,1076
1043,1128
1042,1288
1041,1017
1040,946


In [0]:
%sql
SELECT COUNT(*) FROM f1_processed.lap_times;

count(1)
493054


In [0]:
%sql
DESCRIBE HISTORY f1_processed.lap_times;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-01-21T05:53:47Z,144978267878834,planet.x3@outlook.com,MERGE,"Map(predicate -> [""(((race_id#13475 = race_id#13413) AND (driver_id#13476 = driver_id#13421)) AND (lap#13477 = lap#13373))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1171883731003620),0107-140005-y98qzdej,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 11127, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1217, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 1124, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 1124, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1124, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1188)",null,Databricks-Runtime/16.4.x-scala2.13
4,2026-01-21T05:51:43Z,144978267878834,planet.x3@outlook.com,MERGE,"Map(predicate -> [""(((race_id#12249 = race_id#12187) AND (driver_id#12250 = driver_id#12195)) AND (lap#12251 = lap#12147))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1171883731003620),0107-140005-y98qzdej,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2763319, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3054, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 490904, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 490904, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 490904, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3028)",null,Databricks-Runtime/16.4.x-scala2.13
3,2026-01-21T05:33:36Z,144978267878834,planet.x3@outlook.com,MERGE,"Map(predicate -> [""(((race_id#10862 = race_id#10552) AND (driver_id#10863 = driver_id#10560)) AND (lap#10864 = lap#10512))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1171883731003620),0107-140005-y98qzdej,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 727, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1026, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 701)",null,Databricks-Runtime/16.4.x-scala2.13
2,2026-01-21T05:33:34Z,144978267878834,planet.x3@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""file_date""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1171883731003620),0107-140005-y98qzdej,1,WriteSerializable,false,"Map(numFiles -> 1, numRe